# 06 — The capstone meets Gecko

**Goal:** compose the bootcamp's research assistant with a comprehended API surface: the SAME bounded-tool discipline, one more tool.

Offline version: the 'surface' is a comprehended-tool table derived from the petstore fixture (exactly what notebook 02 built). Live version: the tool calls a served Gecko surface instead — same contract, different edge. **That symmetry is the whole point.**

In [ ]:
# Offline by default. Repo + fixture paths:
from pathlib import Path
import sys

here = Path.cwd()
while not (here / "pyproject.toml").exists():
    here = here.parent
sys.path.insert(0, str(here / "src"))
FIXTURES = here / "cookbook" / "fixtures"
CORPUS_DIR = here / "data" / "corpus"
print(f"fixtures: {FIXTURES}")

## 1. A Gecko-comprehended surface as a bootcamp Tool

In [ ]:
import yaml

from bootcamp_agent.tools import Tool, ToolError

spec = yaml.safe_load((FIXTURES / "petstore-mini.yaml").read_text())

# The comprehended surface (as notebook 02 derived it): operation -> contract.
SURFACE = {
    op['operationId']: {
        'summary': op.get('summary', ''),
        'method': method.upper(),
        'path': path,
        'needs_auth': bool(op.get('security')),
        'params': {p['name']: p.get('schema', {}) for p in op.get('parameters', [])},
    }
    for path, methods in spec['paths'].items()
    for method, op in methods.items()
}


def describe_api_operation(operation_id: str) -> str:
    op = SURFACE.get(operation_id)
    if op is None:
        raise ToolError(
            f'describe_api_operation: unknown operation {operation_id!r}; '
            f'valid: {sorted(SURFACE)}'
        )
    auth = 'requires auth (injected at call time)' if op['needs_auth'] else 'no auth'
    params = ', '.join(f"{k}({v.get('type','?')})" for k, v in op['params'].items()) or '-'
    return f"{op['method']} {op['path']} — {op['summary']} | params: {params} | {auth}"


api_tool = Tool(
    name='describe_api_operation',
    description='Describe one operation of the comprehended Petstore surface by id.',
    run=describe_api_operation,
)
print(api_tool.run(operation_id='placeOrder'))

## 2. Same boundary discipline, proven

The new tool obeys the Session-5 rules: unknown ids get a helpful error, and nothing about auth values ever appears.

In [ ]:
try:
    api_tool.run(operation_id='deleteEverything')
except ToolError as error:
    print(f'refused with guidance: {error}')

assert 'X-Api-Key' not in api_tool.run(operation_id='placeOrder')
print('auth surfaced as a requirement, never as a value')

## 3. The assistant answers an API question — grounded in the surface

Route a question through the comprehended surface the same way the capstone routes through the corpus: retrieve the relevant operation, answer from it, refuse what the surface does not support.

In [ ]:
def answer_api_question(question: str) -> str:
    lowered = question.lower()
    scored = []
    for op_id, op in SURFACE.items():
        text = f"{op_id} {op['summary']} {op['path']}".lower()
        overlap = sum(1 for token in lowered.split() if token in text)
        if overlap:
            scored.append((overlap, op_id))
    if not scored:
        return 'The comprehended surface has no operation for that — refusing rather than inventing one.'
    _, best = max(scored)
    return f'Use {best}: {describe_api_operation(best)}'

print(answer_api_question('how do I list the pets that are available?'))
print(answer_api_question('how do I delete a pet?'))

## 4. Swap the edge for the real thing

Live, the only change is where `SURFACE` comes from: instead of parsing the fixture yourself, your MCP-connected assistant asks the served Gecko surface (notebook 04). The tool contract, the boundary tests, and the refusal behavior are identical — **one code path, the edge swaps**.